# [Wild-GS: Real-Time Novel View Synthesis from Unconstrained Photo Collections](https://arxiv.org/pdf/2406.10373)



Wild-GS is trying to solve one main problem: How can we reconstruct a realistic 3D scene from tourist photographs taken under different lighting conditions, with different cameras, and with temporary objects appearing in the images — without the high computational cost of NeRF?

## 1. Representing a 3D Gaussian

Each Gaussian is defined by its center and covariance matrix.

Center:

$$ \boldsymbol{\mu}\in\mathbb{R}^{3} $$

$$ \boldsymbol{\mu}= \begin{bmatrix} \mu_x\\\\\mu_y\\\\\mu_z \end{bmatrix} $$

Covariance matrix:

$$ \boldsymbol{\Sigma}\in\mathbb{R}^{3\times3} $$

It describes the Gaussian's size, shape, and orientation in 3D space.

The 3D Gaussian density at a point $ \mathbf{x}$ is

$$ G(\mathbf{x})= \exp\left( -\frac12(\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right) $$

where $ \mathbf{x}\in\mathbb{R}^3$.

Meaning: Points near the center have higher Gaussian values; points farther away have lower values. This defines the spatial influence of a Gaussian.

### Decomposing covariance

$$ \boxed{\boldsymbol{\Sigma}=\mathbf{R}\mathbf{S}\mathbf{S}^{T}\mathbf{R}^{T}} $$

Here,

$$ \mathbf{S}= \begin{bmatrix} s_x&0&0\\\ 0&s_y&0\\\ 0&0&s_z \end{bmatrix} $$

- $ \mathbf{S}$: controls the Gaussian's scale along three axes.
- $ \mathbf{R}$: rotation matrix determined by rotation parameters $ r$.

Since covariance must be positive semidefinite,

$$ \mathbf{v}^{T}\boldsymbol{\Sigma}\mathbf{v} =||\mathbf{S}^{T}\mathbf{R}^{T}\mathbf{v}||^2 \geq 0 $$

for any vector $ \mathbf{v}$.

Meaning: The Gaussian can change size and orientation while maintaining a valid covariance matrix.



## 2. Projecting the 3D Gaussian into 2D

The camera projects a 3D Gaussian onto the image plane.

The paper gives:

$$ \boxed{ \boldsymbol{\Sigma}'= \mathbf{J}\mathbf{W}\boldsymbol{\Sigma} \mathbf{W}^{T}\mathbf{J}^{T} } \tag{1} $$

where:

- $ \boldsymbol{\Sigma}$: covariance in world coordinates.
- $ \mathbf{W}$: world-to-camera transformation, with rotation and translation.
- $ \mathbf{J}$: Jacobian of the local affine approximation of perspective projection.
- $ \boldsymbol{\Sigma}'$: projected 2D covariance.

### Derivation

First, transform a 3D point into camera coordinates:

$$ \mathbf{x}_c=\mathbf{R}_c\mathbf{x}+\mathbf{t}_c $$

The covariance transforms under rotation as

$$ \boldsymbol{\Sigma}_c =\mathbf{R}_c\boldsymbol{\Sigma}\mathbf{R}_c^T $$

Translation changes the center but not the covariance.

Next, approximate the perspective projection locally using a first-order Taylor expansion:

$$ \mathbf{u}\approx \pi(\boldsymbol{\mu}_c) +\mathbf{J}(\mathbf{x}_c-\boldsymbol{\mu}_c) $$

For a linear transformation $ \mathbf{y}=\mathbf{A}\mathbf{x}$, covariance follows

$$ \operatorname{Cov}(\mathbf{y}) =\mathbf{A}\operatorname{Cov}(\mathbf{x})\mathbf{A}^T $$

Using $ \mathbf{A}=\mathbf{J}\mathbf{R}_c$, we obtain

$$ \boldsymbol{\Sigma}' =\mathbf{J}\mathbf{R}_c\boldsymbol{\Sigma} \mathbf{R}_c^T\mathbf{J}^T $$

This is the covariance-projection equation in the paper, with $ \mathbf{W}$ representing the relevant linear transformation.

Meaning: A 3D ellipsoid becomes a 2D elliptical footprint on the image. The renderer can then rasterize that footprint efficiently.


## 3. Computing the color of a pixel: Alpha compositing

Each Gaussian stores an opacity $ \alpha_i$ and spherical harmonic (SH) coefficients used to compute its view-dependent color $ \mathbf{c}_i$.

When projected onto the image, Gaussian $ i$ contributes an effective opacity

$$ \boxed{\alpha_i'=\alpha_i G_i^{2D}(\mathbf{p})} $$

where $ G_i^{2D}(\mathbf{p})$ is its 2D Gaussian value at pixel $ \mathbf{p}$.

The paper's color equation is

$$ \boxed{ \mathbf{C}= \sum_{i=1}^{n} \mathbf{c}_i\alpha_i' \prod_{j=1}^{i-1}(1-\alpha_j') } \tag{2} $$

Gaussians are sorted from front to back by depth.

### Derivation

For the first Gaussian, the contribution is

$$ \mathbf{C}_1=\mathbf{c}_1\alpha_1' $$

The fraction of light not covered by it is

$$ T_2=1-\alpha_1' $$

Therefore, the second Gaussian contributes

$$ \mathbf{C}_2=\mathbf{c}_2\alpha_2'(1-\alpha_1') $$

The third Gaussian contributes

$$ \mathbf{C}_3= \mathbf{c}_3\alpha_3' (1-\alpha_1')(1-\alpha_2') $$

In general, the transmittance reaching Gaussian $ i$ is

$$ T_i=\prod_{j=1}^{i-1}(1-\alpha_j') $$

Hence, its color contribution is $ \mathbf{c}_i\alpha_i'T_i$. Adding all contributions gives Equation (2).

Meaning: A Gaussian closer to the camera can obscure Gaussians behind it. Fully opaque front Gaussians prevent further Gaussians from contributing at that pixel.




## 4. Spherical harmonics (SH)

The color of a Gaussian can vary with the viewing direction $ \mathbf{d}$.

Conceptually,

$$ \boxed{ \mathbf{c}_i(\mathbf{d}) =\sum_{k=1}^{K}\mathbf{a}_{ik}Y_k(\mathbf{d}) } $$

where:

- $ Y_k(\mathbf{d})$: spherical harmonic basis functions.
- $ \mathbf{a}_{ik}$: learnable coefficients for Gaussian $ i$.
- $ K$: number of basis functions.

Meaning: SH coefficients allow a Gaussian to represent different appearances when viewed from different directions, which helps model view-dependent effects.

## 5. Densification and pruning

These are adaptive operations rather than equations specified in the provided passage.

- Densification: split a Gaussian or add Gaussians in regions where more detail is needed.
- Pruning: remove Gaussians that contribute little, such as unnecessary or nearly transparent ones.

Purpose: allocate representation capacity where it improves reconstruction while keeping the scene efficient.

The main idea of a **triplane representation** is to represent 3D spatial information using three 2D feature maps instead of storing a full 3D feature grid.

1. Mathematical representation
A triplane consists of three 2D feature planes:

$$ P_{XY},\quad P_{YZ},\quad P_{ZX} $$

Each plane stores feature vectors at different 2D coordinates.
Suppose a 3D point is

$$ \mathbf{p}=(x,y,z) $$

Project this point onto each plane:

$$ \begin{aligned}
P_{XY}&:\ (x,y)\\
P_{YZ}&:\ (y,z)\\
P_{ZX}&:\ (z,x)
\end{aligned} $$

Query the corresponding feature vectors using interpolation:

$$ \mathbf{f}_{XY}=\operatorname{Sample}(P_{XY},x,y) $$


$$ \mathbf{f}_{YZ}=\operatorname{Sample}(P_{YZ},y,z) $$


$$ \mathbf{f}_{ZX}=\operatorname{Sample}(P_{ZX},z,x) $$

Then combine them:

$$ \boxed{\mathbf{f}_{p}=
\mathbf{f}_{XY}+\mathbf{f}_{YZ}+\mathbf{f}_{ZX}} $$

Here, $\mathbf{f}_p$ is the feature vector representing the 3D point.

Numerical example
Suppose each plane stores a feature vector with two components. For a particular 3D point, interpolation gives:

$$ \mathbf{f}_{XY}=[1,2] $$


$$ \mathbf{f}_{YZ}=[3,1] $$


$$ \mathbf{f}_{ZX}=[2,4] $$

Adding the features:

$$ \mathbf{f}_{p}=[1,2]+[3,1]+[2,4] $$


$$ \boxed{\mathbf{f}_{p}=[6,7]} $$

- The resulting vector combines information from all three coordinate planes.
- Intuition: The XY plane tells us about the point's location in the horizontal plane, the YZ plane provides another view of its spatial context, and the ZX plane provides the third.
- Combining their features gives a representation of the 3D point.

3. Why use three 2D planes instead of a 3D grid?
Suppose we use a grid with resolution $N$along each dimension.


A full 3D feature grid requires approximately

$$ N^3F $$
feature values, where $F$is the number of features per grid location.
A triplane requires approximately

$$ 3N^2F $$
feature values.
For $N=256$and $F=32$ :
- 3D grid: $256^3\times32=536{,}870{,}912$values.
- Triplane: $3\times256^2\times32=6{,}291{,}456$ values.


Thus, the triplane uses approximately 85 times fewer feature values in this example.

The trade-off is that a triplane combines projected features rather than directly storing an independent feature vector at every 3D grid location.

What does the MLP do?
In the original triplane approach described in your paper, the combined feature is passed to a multilayer perceptron (MLP):
$$ \mathbf{f}_{p}
\longrightarrow
\operatorname{MLP}
\longrightarrow
(\sigma,\mathbf{c}) $$

where:
- $\sigma$: density, indicating how much matter or scene content is present at that point.
- $\mathbf{c}$: predicted color.


These outputs are used by volume rendering to generate an image.

The MLP learns how to interpret the feature vector.

The triplane provides the spatial features;

The MLP decodes them into useful rendering quantities.

# Wild-GS aims to make 3D Gaussian Splatting robust to changing appearances and transient objects in tourist photographs while retaining fast training and rendering.

It has three main components:

1\. **2D Parsing Module**: Processes reference images to extract global appearance information and predict masks that identify static scene regions while excluding transient objects.

2\. **3D Wrapping Module**: Maps 3D Gaussian positions to local appearance embeddings using triplane features, associating detailed image appearance with the corresponding 3D locations.

3\. **Fusion Network**: Combines global appearance embeddings, local appearance embeddings, and intrinsic Gaussian features to predict adaptive spherical harmonic (SH) coefficients for each Gaussian, enabling appropriate color rendering under different viewing conditions.

Overall pipeline:

Reference images → 2D appearance features and masks → 3D-aware local features → feature fusion → SH prediction → rendered image.

The key idea is to separate global appearance, local appearance, and intrinsic Gaussian properties, allowing Wild-GS to handle appearance variations without abandoning efficient 3DGS rendering.

# 4.1 Hierarchical Appearance Modeling — Full mathematical derivation

The objective is to predict the appearance of each 3D Gaussian by combining three sources of information: global appearance, local appearance, and intrinsic material features.

## 1. Define the three appearance components

Suppose the scene contains $N$ Gaussians, indexed by $i=1,\ldots,N$.

### (a) Global appearance embedding

$$ \mathbf{E}^{g}\_{mbg}\in\mathbb{R}^{d_g} $$

This represents scene-wide appearance information for a reference view, such as illumination level and tone mapping.

For example, if one photograph is bright and another is dark, the global embedding can encode that difference.

### (b) Local appearance embedding

$$ \mathbf{E}^{l}\_{mbg,i}\in\mathbb{R}^{d_l} $$

This represents the local, position-dependent appearance of Gaussian $i$, including variations in local reflectance.

Two Gaussians in the same scene can have different local embeddings because they correspond to different surfaces or locations.

### (c) Intrinsic feature

$$ \mathbf{f}^{in}\_i\in\mathbb{R}^{d_f} $$

This stores inherent attributes associated with the material or dominant region represented by Gaussian $i$.

For example, it may help distinguish the appearance characteristics of stone from those of glass.

The exact physical meaning of these learned features depends on how the network is trained; they are not necessarily direct measurements of real-world material properties.

## 2. Concatenate the three components

The paper uses $\oplus$ to denote concatenation.

$$ \boxed{ \mathbf{z}\_i = \mathbf{E}^{g}\_{mbg} \oplus \mathbf{E}^{l}\_{mbg,i} \oplus \mathbf{f}^{in}\_i } $$

If the dimensions are $d_g,d_l,d_f$, then

$$ \mathbf{z}\_i\in\mathbb{R}^{d_g+d_l+d_f} $$

For example, suppose

$$ \mathbf{E}^{g}\_{mbg}=[0.8,0.3] $$

$$ \mathbf{E}^{l}\_{mbg,i}=[0.2,0.9] $$

$$ \mathbf{f}^{in}\_i=[0.5,0.4] $$

Concatenation gives

$$ \boxed{\mathbf{z}\_i=[0.8,0.3,0.2,0.9,0.5,0.4]} $$

Notice that concatenation does not add the vectors element by element. It preserves the components in a single longer vector.

## 3. Pass the combined feature through the fusion network

Equation (3) in the paper is

$$ \boxed{ \mathbf{sh}\_i = M^F\_\theta \left( \mathbf{E}^{g}\_{mbg} \oplus \mathbf{E}^{l}\_{mbg,i} \oplus \mathbf{f}^{in}\_i \right) } \tag{3} $$

Using the combined feature vector, this becomes

$$ \mathbf{sh}\_i=M^F\_\theta(\mathbf{z}\_i) $$

Here:

- $M^F\_\theta$ is the shared fusion network.
- $\theta$ represents its learnable parameters.
- $\mathbf{sh}\_i$ is the predicted set of spherical harmonic coefficients for Gaussian $i$.

To understand the mathematics, suppose the fusion network is an illustrative two-layer MLP:

$$ \mathbf{h}\_i= \operatorname{ReLU} (\mathbf{W}\_1\mathbf{z}\_i+\mathbf{b}\_1) $$

$$ \boxed{ \mathbf{sh}\_i= \mathbf{W}\_2\mathbf{h}\_i+\mathbf{b}\_2 } $$

This is an example of how an MLP can implement the fusion function, not a claim that these are the paper's exact layer specifications.

The weights and biases are learned during training so that the predicted appearance helps the rendered images match the reference images.

## 4. What does the output $\mathbf{sh}\_i$ mean?

The output is not necessarily a single RGB color. It is a set of SH coefficients used to calculate the Gaussian's view-dependent color.

For a viewing direction $\mathbf{d}$, the color can be expressed conceptually as

$$ \boxed{ \mathbf{c}\_i(\mathbf{d}) = \sum\_{k=1}^{K} \mathbf{sh}\_{i,k}Y_k(\mathbf{d}) } $$

where:

- $\mathbf{sh}\_{i,k}$ is the $k$-th SH coefficient, potentially with RGB components.
- $Y_k(\mathbf{d})$ is the corresponding spherical harmonic basis function.
- $K$ is the number of basis functions.

Thus, the fusion network predicts coefficients, and the rendering process evaluates those coefficients for the relevant viewing direction.

## 5. How the three components work together

Suppose Gaussian $i$ represents a stone surface.

| Component         | Example information                     | Role                                  |
| ----------------- | --------------------------------------- | ------------------------------------- |
| Global embedding  | Photograph is warm and brightly exposed | Adapts scene-wide appearance          |
| Local embedding   | This part of the wall is in shadow      | Adapts local appearance               |
| Intrinsic feature | Surface has stone-like characteristics  | Supplies inherent feature information |

The fusion network combines all three to predict the Gaussian's SH coefficients.

This decomposition helps the model distinguish a scene-wide change from a local change, rather than forcing one appearance representation to explain everything.

## 6. Why does Wild-GS preserve rendering efficiency?

Standard 3DGS already uses SH coefficients to calculate Gaussian colors. Wild-GS predicts those coefficients using its fusion network.

Once the coefficients have been computed and cached for each Gaussian, rendering can proceed through the usual 3DGS rasterization pipeline.

The central efficiency idea is therefore:

$$ \text{Appearance features} \rightarrow \text{Fusion network} \rightarrow \text{Cached SH coefficients} \rightarrow \text{Standard rasterization} $$

The network adds computation when appearance coefficients need to be generated or updated, but it does not require replacing the original Gaussian rasterizer with a large neural rendering network for every pixel.

Final takeaway: Equation (3) is a learned mapping from three complementary feature sources to the SH coefficients of each Gaussian. Global features capture scene-wide appearance, local features capture spatial variations, and intrinsic features provide material-related information. Their fusion gives each Gaussian a more adaptable appearance while retaining the fast rendering mechanism of 3DGS.

**Hierarchical Appearance Modeling in Wild-GS **solves the problem of making the same 3D scene look consistent when photographs have different lighting, exposure, and local surface appearances.
- It determines the appearance of every 3D Gaussian using three types of information:
1. Global appearance embedding: Captures scene-wide changes, such as an image being brighter, darker, or warmer in color.
2. Local appearance embedding: Captures appearance differences at specific locations, such as one part of a wall being in shadow while another is illuminated.
3. Intrinsic features: Store learned information about the characteristics of the surface or material represented by each Gaussian.
- A shared fusion network combines these three inputs and predicts the Gaussian's spherical harmonic (SH) coefficients, which are used to determine its rendered color, including view-dependent appearance.


**Why is this useful?**
- Without this decomposition, the model may struggle to distinguish global lighting changes from local surface differences.
- Wild-GS gives it separate sources of information for handling each.

**How does it remain fast?**
- Once the SH coefficients are predicted and cached, Wild-GS uses the original 3DGS rasterization process instead of requiring the fusion network to run for every pixel during rendering.

**Global appearance information cannot capture fine, location-specific appearance changes, so Wild-GS assigns local appearance features to individual 3D Gaussians.**

For example, a building may have a bright wall, a shadowed corner, and a shiny window. The global appearance embedding describes the overall image, but it cannot adequately describe all these local differences.


**Wild-GS solves this by combining triplane feature maps with the 3D Gaussian representation.**
1. The reference image is processed to extract appearance features and construct triplane feature maps.
2. Each 3D Gaussian has a learned 3D center position.
3. The Gaussian uses its position to query the corresponding features from the three triplane planes.
4. The sampled features are combined to form that Gaussian's local appearance embedding.
5. This local embedding is later combined with global appearance information and intrinsic features to predict its SH coefficients.


**Why is this important?**

- It establishes a spatial correspondence between image appearance and 3D Gaussians.

- Instead of applying the same appearance adjustment to the entire scene,

- Wild-GS can adapt the appearance of individual regions, preserving fine details such as local shadows, highlights, and texture variations.

**4.1.2 Local Appearance Control — Triplane Color Creation**


The overall objective is to take a reference photograph, remove transient objects, reconstruct its visible 3D points, convert their colors into three orthogonal planes, and use those planes to give each 3D Gaussian a local appearance embedding.
Step 1: Rendered depth — Equation (5)
The paper defines the rendered depth as

$$ \boxed{
\hat D_{I_R}
=
\sum_{i=1}^{n}
d_i\alpha_i'
\prod_{j=1}^{i-1}(1-\alpha_j')
}
\tag{5} $$

Define each variable
- $ I_R $: reference image.
- $ \hat D_{I_R} $: rendered depth map for that image.
- $ n $: number of Gaussians contributing to a pixel.
- $ d_i $: depth of Gaussian $ i $ from the camera.
- $ \alpha_i' $: effective opacity of Gaussian $ i $ at the pixel.
- $ \prod_{j=1}^{i-1}(1-\alpha_j') $: transmittance through all preceding Gaussians.
Recall from standard 3DGS:

$$ \alpha_i'=\alpha_iG_i^{2D}(\mathbf{p}) $$

where $ \alpha_i $ is the Gaussian's opacity and $ G_i^{2D}(\mathbf{p}) $ is its projected 2D Gaussian value at pixel $ \mathbf{p} $.

**Derivation**


The first Gaussian contributes depth

$$ d_1\alpha_1' $$

The second Gaussian contributes only the fraction not covered by the first:

$$ d_2\alpha_2'(1-\alpha_1') $$

The third Gaussian contributes

$$ d_3\alpha_3'(1-\alpha_1')(1-\alpha_2') $$

Therefore, the total rendered depth is the weighted sum of all Gaussian depths:

$$ \hat D_{I_R}
=
d_1\alpha_1'
+d_2\alpha_2'(1-\alpha_1')
+\cdots $$

This gives Equation (5).
Meaning: each pixel receives a depth estimate based on the depths and visibility of the Gaussians contributing to it. This depth map is then used to reconstruct 3D points from the reference image.


Note that this is an alpha-weighted depth sum; the equation shown does not explicitly divide by the accumulated opacity.

**Step 2: Back-project the image into 3D**

Equation (6)
The paper gives

$$ \boxed{
\{C_{I_R},P_{I_R}\}
=
BP\left(
I_R[M_{I_R}>Th],
\hat D_{I_R}[M_{I_R}>Th],
\omega_{I_R}
\right)
}
\tag{6}
$$

Here, $BP$ means back-projection.
- $C_{I_R}$: colors of the reconstructed 3D points.
- $P_{I_R}$: 3D positions of those points.
- $M_{I_R}$: visibility mask.
- $Th$: threshold used to distinguish static from transient regions.
- $\omega_{I_R}$: camera parameters, including the relevant camera intrinsics and pose.


The notation $M_{I_R}>Th$ selects pixels considered valid for reconstruction.

The image and depth map are filtered using the same mask, so their pixels remain aligned.

**2.1 How does back-projection work mathematically?**

Suppose a pixel has image coordinates $(u,v)$, depth $d$, and camera intrinsic matrix

$$ K=
\begin{bmatrix}
f_x&0&c_x\\
0&f_y&c_y\\
0&0&1
\end{bmatrix} $$

The pixel in homogeneous coordinates is

$$ \tilde{\mathbf{u}}=
\begin{bmatrix}
u\\v\\1
\end{bmatrix} $$

Back-project it into camera coordinates:

$$ \boxed{
\mathbf{p}_c=dK^{-1}\tilde{\mathbf{u}}
} $$

Expanding the result gives

$$ x_c=\frac{(u-c_x)d}{f_x} $$


$$ y_c=\frac{(v-c_y)d}{f_y} $$


$$ z_c=d $$

If $R_{wc}$ and $\mathbf{t}_{wc}$ transform camera coordinates into world coordinates, then

$$ \boxed{
\mathbf{p}_w=R_{wc}\mathbf{p}_c+\mathbf{t}_{wc}
} $$

The RGB value of the original pixel is assigned to this 3D point.
Thus, repeating this operation for all valid pixels constructs a colored point cloud:

$$ P_{I_R}=\{\mathbf{p}_1,\mathbf{p}_2,\ldots,\mathbf{p}_m\} $$


$$ C_{I_R}=\{\mathbf{c}_1,\mathbf{c}_2,\ldots,\mathbf{c}_m\} $$

where $m$ is the number of selected pixels.

**Why the visibility mask matters:**

- if a tourist covers a building in the reference image, the corresponding pixels should not be treated as reliable samples of the permanent scene.

- The mask excludes those pixels before back-projection.

- One important detail: back-projection reconstructs the visible surface using the estimated depth. It does not recover hidden surfaces that the camera cannot see.

**Step 3: Normalize the 3D points and project them onto three planes**

The paper writes
$$
\boxed{
\{c_{xy},c_{yz},c_{zx}\}
=
Proj(C_{I_R},\tilde P_{I_R})
}
\tag{7}
$$

where:
- $P_{I_R}$: reconstructed 3D point positions.
- $\tilde P_{I_R}$: normalized 3D positions.
- $C_{I_R}$: point colors.
- $c_{xy},c_{yz},c_{zx}$: projected color maps on the three orthogonal planes.

**3.1 Why normalize coordinates?**

- The reconstructed points may occupy arbitrary world-coordinate ranges.
- The triplane needs coordinates that map consistently onto finite 2D feature maps.
- A simple normalization for a coordinate $x$ within a known interval $[x_{\min},x_{\max}]$ is

$$ \boxed{
\tilde x=
2\frac{x-x_{\min}}{x_{\max}-x_{\min}}-1
} $$

This maps the interval to $[-1,1]$. The same procedure can be applied to $y$ and $z$.
The paper's exact normalization details depend on its implementation; the equation above illustrates the standard mapping.

 **3.2 Project the normalized coordinates
For a normalized point**

$$ \tilde{\mathbf p}=(\tilde x,\tilde y,\tilde z) $$

the three projections are

$$ P_{XY}:\quad(\tilde x,\tilde y) $$


$$ P_{YZ}:\quad(\tilde y,\tilde z) $$


$$ P_{ZX}:\quad(\tilde z,\tilde x) $$

The point's color is splatted or accumulated at the corresponding 2D location on each plane.
For example, a red point with normalized coordinates

$$ \tilde{\mathbf p}=(0.2,-0.4,0.6) $$

is projected onto:
- XY plane: $(0.2,-0.4)$, colored red.
- YZ plane: $(-0.4,0.6)$, colored red.
- ZX plane: $(0.6,0.2)$, colored red.
Repeating this for the colored point cloud creates three 2D color maps.

**3.3 Why project along the reverse axes too?**
- A single projection direction can lose information when different 3D points overlap on the same 2D plane.
- For example, projecting along the Z axis collapses depth: two points with the same X and Y coordinates but different Z coordinates can occupy the same XY pixel.
- Wild-GS therefore also constructs reverse-axis projections. Conceptually, the model gets complementary views of the scene instead of relying on only one projection direction.

**Step 4: Create high-dimensional triplane features — Equation (8)**
The paper defines

$$ \boxed{
F^T_{I_R}
=
U^ {3D}_{\theta}
\left(
\{c_{xy},c_{yz},c_{zx}\}
\oplus
\{c'_{xy},c'_{yz},c'_{zx}\}
\right)
}
\tag{8} $$

Here:
- $c_{xy},c_{yz},c_{zx}$: forward-projection color maps.
- $c'_{xy},c'_{yz},c'_{zx}$: reverse-projection color maps.
- $\oplus$: concatenation.
- $U^{3D}_{\theta}$: the trainable U-Net used to process the projected color maps.
- $F^T_{I_R}$: the resulting triplane feature maps.
Derivation
First: combine the six color maps by concatenation:

$$ C^T=
c_{xy}\oplus c_{yz}\oplus c_{zx}
\oplus c'_{xy}\oplus c'_{yz}\oplus c'_{zx} $$

Concatenation preserves the information from each projection rather than adding their RGB values together.
Second: process these maps using the U-Net:

$$ \boxed{F^T_{I_R}=U^{3D}_{\theta}(C^T)} $$

A U-Net uses convolutional layers to extract spatial features at different scales.

Its learned filters can identify patterns such as edges, texture, and relationships between neighboring regions.

The output contains feature vectors rather than merely the original RGB colors.

Each plane can now encode richer appearance information.


**Why is this needed?**

The original projected color maps are relatively simple.

The U-Net transforms them into learned features that are more useful for representing complex local appearance.

**Derivation**

**First:** combine the six color maps by concatenation:

$$ C^T=
c_{xy}\oplus c_{yz}\oplus c_{zx}
\oplus c'_{xy}\oplus c'_{yz}\oplus c'_{zx} $$

Concatenation preserves the information from each projection rather than adding their RGB values together.

**Second:** process these maps using the U-Net:

$$ \boxed{F^T_{I_R}=U^{3D}_{\theta}(C^T)} $$

A U-Net uses convolutional layers to extract spatial features at different scales.

Its learned filters can identify patterns such as edges, texture, and relationships between neighboring regions.


The output contains feature vectors rather than merely the original RGB colors.

Each plane can now encode richer appearance information.

**Why is this needed?**

The original projected color maps are relatively simple. The U-Net transforms them into learned features that are more useful for representing complex local appearance.

**Step 5: Query the triplane features for each Gaussian**


Suppose Gaussian $i$ has learned center

$$ \boldsymbol{\mu}_i=(x_i,y_i,z_i) $$

Normalize its coordinates using the same coordinate system as the triplanes, then project it onto each plane.
Sample the feature maps at the corresponding locations:

$$ \mathbf f^i_{xy}
=
\operatorname{Sample}(F^T_{xy},x_i,y_i) $$


$$ \mathbf f^i_{yz}
=
\operatorname{Sample}(F^T_{yz},y_i,z_i) $$


$$ \mathbf f^i_{zx}
=
\operatorname{Sample}(F^T_{zx},z_i,x_i) $$

Here, $\operatorname{Sample}$ typically uses bilinear interpolation to obtain a feature vector even when the Gaussian's projected location falls between grid cells.
Combine the three features:

$$ \boxed{
\mathbf f^i_{\mathrm{local}}
=
\mathbf f^i_{xy}
+
\mathbf f^i_{yz}
+
\mathbf f^i_{zx}
} $$

The sum combines information from all three planes into one feature vector.

 **Step 6: Generate the local appearance embedding**

$$ \boxed{
E^l_{mbg,i}
=
ML^L_{\theta}
\left(
\mathbf f^i_{xy}
+
\mathbf f^i_{yz}
+
\mathbf f^i_{zx}
\right)
} $$

where $ML^L_{\theta}$  is a trainable MLP.
The MLP transforms the combined triplane feature into the local appearance embedding for Gaussian $i$ .

**Overall pipeline**



1.   Reference image + rendered depth
        - Equations (5)–(6): depth rendering and back-projection

1.   Static colored 3D point cloud
        - Mask out transient regions

1.   Three forward and three reverse projections
        - Equation (7): build triplane color maps
2.   U-Net
        - Equation (8): convert color maps into feature maps


2.   Sample at each Gaussian's position
        - Interpolate three plane features and sum them


2.   MLP → Local appearance embedding
        - Used to predict adaptive SH coefficients


















# Efficient Triplane Sampling.

**computational problem: high-resolution triplane feature maps are expensive to sample, but most 3D Gaussians occupy only a small region of those maps.**

**How Wild-GS solves it**

1. Find the region containing most Gaussians. Use an axis-aligned bounding box (AABB), defined by minimum and maximum coordinates along X, Y, and Z.
2. Restrict triplane sampling to that region. Instead of allocating sampling resources across the entire spatial extent, Wild-GS focuses on the region where Gaussians are concentrated. This improves feature-map utilization and reduces sampling costs.
3. Handle Gaussians outside the box. Rather than sampling the triplane, assign those Gaussians a shared learnable embedding vector, $ v $:
$$ \boxed{\mathbf E^l_{mbg,i}=\mathbf v} $$

   The vector is learned during training and provides a default local appearance representation for these Gaussians.

What is an AABB?
An AABB is a 3D box whose faces are aligned with the coordinate axes. A point \((x,y,z)\) lies inside it if

$$ x_{\min}\le x\le x_{\max} $$


$$ y_{\min}\le y\le y_{\max} $$


$$ z_{\min}\le z\le z_{\max} $$

Gaussians outside these limits receive the default embedding instead of querying the triplane.

**Why is this useful?**

High-resolution feature maps preserve detailed appearance, but sampling every Gaussian across the entire spatial region wastes computation when the Gaussians are concentrated.

The AABB focuses sampling on the useful region, while the learnable vector provides a fallback for points outside it.

**Important distinction: the AABB reduces the region used for triplane sampling; it does not necessarily remove Gaussians outside the box from the 3D scene or from rendering.**

**4.1.3 Learnable Intrinsic Feature** — Explained


- This section introduces a third feature that describes what a Gaussian inherently looks like, separate from lighting and environmental effects.
- Recall that Wild-GS models appearance using three components:
1. Global appearance embedding — captures scene-wide lighting, exposure, and color tone.
2. Local appearance embedding — captures spatially varying appearance, such as local reflections or color changes.
3. Intrinsic feature — captures the Gaussian's learned, material-related appearance characteristics

**1. What does intrinsic appearance mean?**

Imagine a red car photographed under different conditions.

Wild-GS tries to separate these two things:
- External appearance: changes caused by lighting, exposure, and viewing conditions.
- Intrinsic appearance: learned characteristics associated with the object's material and underlying appearance.

Importantly, this is a learned feature, not a guaranteed physically accurate measurement of the material.

**2. How is it represented mathematically?**

For each Gaussian $i $, Wild-GS learns an intrinsic feature vector:


 $$ \mathbf f_i^{\mathrm{in}}\in\mathbb R^d $$

Here:
- $i $ identifies the Gaussian.
- $d $ is the feature-vector dimension.
- $\mathbf f_i^{\mathrm{in}} $ is optimized during training.


This vector is a compact latent representation. Its individual dimensions do not necessarily correspond directly to physical properties such as roughness, metallicity, or albedo.


**3. How does it work with the other features?**
Wild-GS combines all three appearance components:



 $$ \mathbf z_i =
\mathbf E^{g}_{mbg}
\oplus
\mathbf E^{l}_{mbg,i}
\oplus
\mathbf f_i^{\mathrm{in}} $$

The fusion network then predicts the Gaussian's spherical-harmonic (SH) coefficients:

 $$ \mathbf{sh}_i=M_\theta^F(\mathbf z_i) $$

Those coefficients are used to render the Gaussian's view-dependent color.


**Why is this useful?**

If the model relied only on image-derived appearance embeddings, it could struggle to distinguish stable object characteristics from temporary lighting and exposure changes.

The intrinsic feature provides an additional learned representation that helps the model represent these factors separately.


**4. What is the connection to EAGLES?**

The authors draw inspiration from EAGLES, which uses compact latent vectors to represent Gaussian attributes.

Wild-GS applies this general idea to intrinsic appearance:

instead of describing every Gaussian only through its image-conditioned appearance, it gives each Gaussian its own learnable feature.


**In one sentence: The intrinsic feature is a per-Gaussian learned memory of its underlying appearance, which complements the global and local embeddings to improve rendering across different photographic conditions.**

# 4.2 Depth Regularization — Full mathematical derivation


The goal of this section is to make the reconstructed 3D geometry more accurate by comparing rendered depth with estimated depth.


Wild-GS uses rendered depth for back-projecting image pixels into 3D points.

If the rendered depth is wrong, the generated 3D point cloud will also be wrong. Depth regularization helps constrain this geometry.


**1. Understand the variables**


The paper uses:
- $ \hat D_{I_R}$: depth map rendered by Wild-GS for reference image $ I_R$.
- $ D^{Est}_{I_R}$: depth estimated from the reference image using Depth Anything.
- $ M_{I_R}$: mask that identifies static image regions and transient objects.
- $ Th$: threshold used to select valid, non-transient pixels.
The notation

$$ \hat D_{I_R}[M_{I_R}>Th] $$

means: select rendered-depth values only at pixels whose mask value exceeds the threshold.
Similarly,

$$ D^{Est}_{I_R}[M_{I_R}>Th] $$

selects the corresponding estimated-depth values at those same pixels.


This masking matters because people, vehicles, and other transient objects can appear in photographs without belonging to the persistent scene geometry.

**2. Start with the Pearson correlation formula**


Let the two depth maps, after masking, be vectors containing $ N$ valid pixel values:

$$ \mathbf d_r=(d_{r,1},d_{r,2},\ldots,d_{r,N}) $$


$$ \mathbf d_e=(d_{e,1},d_{e,2},\ldots,d_{e,N}) $$

Here, $ d_{r,k}$ is the rendered depth at valid pixel $ k$, and $ d_{e,k}$ is the estimated depth at that same pixel.
Pearson correlation measures how strongly two sets of values vary together:

$$ \rho=\frac{\operatorname{Cov}(\mathbf d_r,\mathbf d_e)}
{\sqrt{\operatorname{Var}(\mathbf d_r)
\operatorname{Var}(\mathbf d_e)}} $$

Now we derive each part.

**3. Derive the covariance**


First, calculate the mean rendered depth:

$$ \bar d_r=\frac{1}{N}\sum_{k=1}^{N}d_{r,k} $$

Calculate the mean estimated depth:

$$ \bar d_e=\frac{1}{N}\sum_{k=1}^{N}d_{e,k} $$

The covariance measures whether deviations from the two means tend to have the same sign:

$$ \operatorname{Cov}(\mathbf d_r,\mathbf d_e)
=\frac{1}{N}\sum_{k=1}^{N}
(d_{r,k}-\bar d_r)(d_{e,k}-\bar d_e) $$

Interpretation:
- Both depths tend to increase together: covariance is positive.
- One increases while the other decreases: covariance is negative.
- They show little linear relationship: covariance is near zero.


The normalization convention may use $N-1$ instead of $N$; this does not change Pearson correlation when the same convention is used consistently.

 **4. Derive the variance terms
Rendered-depth variance:**

$$ \operatorname{Var}(\mathbf d_r)
=\frac{1}{N}\sum_{k=1}^{N}(d_{r,k}-\bar d_r)^2 $$

Estimated-depth variance:

$$ \operatorname{Var}(\mathbf d_e)
=\frac{1}{N}\sum_{k=1}^{N}(d_{e,k}-\bar d_e)^2 $$

Variance measures how spread out each depth map's values are around its mean.
Taking the square root of their product gives the normalization term:

$$ \sqrt{\operatorname{Var}(\mathbf d_r)
\operatorname{Var}(\mathbf d_e)} $$

Why divide by this? Covariance depends on the scale of the depth values. Dividing by the standard deviations produces a normalized correlation coefficient, typically between $ -1 $ and $ 1 $.

**5. Substitute everything into the paper's equation**


Substitute the covariance and variances into Pearson correlation:

$$
\boxed{
\rho_D=
\frac{
\frac{1}{N}\sum_{k=1}^{N}
(d_{r,k}-\bar d_r)(d_{e,k}-\bar d_e)
}{
\sqrt{
\left[\frac{1}{N}\sum_{k=1}^{N}(d_{r,k}-\bar d_r)^2\right]
\left[\frac{1}{N}\sum_{k=1}^{N}(d_{e,k}-\bar d_e)^2\right]
}
}
} $$

This is the expanded form of the equation in your paper, with the mask applied implicitly by including only valid pixels.

**6. Numerical example**
Suppose the estimated and rendered depths at three valid pixels are:


\begin{array}{c|c|c}
\text{Pixel} & \text{Estimated Depth} & \text{Rendered Depth} \\
\hline
1 & 1 & 2 \\
2 & 2 & 4 \\
3 & 3 & 6
\end{array}



Their means are:

$$ \bar d_e=2,\qquad \bar d_r=4 $$

Covariance:

$$ \begin{aligned}
\operatorname{Cov}
&=\frac{(2-4)(1-2)+(4-4)(2-2)}{}\\
&\quad+\frac{(6-4)(3-2)}{3}\\
&=\frac{2+0+2}{3}=\frac43
\end{aligned} $$

The rendered-depth variance is $8/3$, and the estimated-depth variance is $2/3$. Therefore,

$$ \rho_D=
\frac{4/3}{\sqrt{(8/3)(2/3)}}=1 $$

The correlation is $1$, because rendered depth is exactly twice the estimated depth at every pixel. The absolute values differ, but their relative depth pattern is identical.

**Important detail: is this really a loss?**


The equation you pasted is Pearson correlation, not a conventional minimization loss by itself.
- If the training objective maximizes this correlation, higher is better.
- If the training objective minimizes a loss, the corresponding expression would typically be:

$$ \boxed{\mathcal L_D=1-\rho_D} $$

Equivalently, minimizing $-\rho_D$ maximizes the correlation.

Check the paper's exact objective to see which convention it uses; the displayed equation alone does not establish the sign.


Also, monocular depth estimates can have scale ambiguity. Correlation helps align the relative depth pattern without requiring the rendered depths to match the estimator's absolute scale exactly.

**Final intuition**
Wild-GS renders a depth map, compares its valid pixels against Depth Anything's estimated depth, and encourages their spatial depth patterns to agree.

This improves the geometry used to reconstruct the 3D point cloud, while masking transient objects prevents temporary objects from distorting the scene geometry.

# 4.3 Handling Transient Objects — Full Mathematical Explanation

## 1. Visibility mask prediction

The reference image is denoted by $I_R$, and the rendered image is denoted by $\hat{I}\_R$.

A 2D parsing module uses a U-Net to predict a visibility mask:

\\[ M\_{IR}=U\_{\theta}^{2D}(I_R) \\]

where:

- $I_R$: Reference image.
- $U\_{\theta}^{2D}$: 2D U-Net that predicts the visibility mask.
- $M\_{IR}$: Predicted visibility mask.
- $\hat{I}\_R$: Rendered image generated by the 3D Gaussian Splatting model.

The mask identifies pixels corresponding to static objects that should contribute to scene reconstruction.

For example, static objects include buildings and roads, while transient objects may include moving cars and pedestrians.

## 2. Image rendering loss

The paper defines the image loss as:

$$
\boxed{
\begin{aligned}
\mathcal{L}_I ={}& \lambda_I \left| I_R \odot M_{IR} - \hat{I}_R \odot M_{IR} \right| \\
& + (1 - \lambda_I) \cdot \operatorname{SSIM} \left( I_R \odot M_{IR}, \hat{I}_R \odot M_{IR} \right)
\end{aligned}
} \tag{11}
$$

Here:

- $\mathcal{L}\_I$: Image reconstruction loss.
- $\lambda_I$: Weight balancing the pixel-wise difference and SSIM term.
- $\odot$: Element-wise multiplication.
- $|\cdot|$: Typically the absolute pixel-wise difference, aggregated over the image.
- $\operatorname{SSIM}$: Structural Similarity Index Measure, which compares image structure, contrast, and luminance.

Important: In conventional optimization, SSIM is often converted into a loss such as $1-\operatorname{SSIM}$. The equation you supplied uses SSIM directly, so I have preserved the paper's expression rather than silently changing it.

### How the mask works

Suppose a pixel has a visibility value of $1$. It is retained in the rendering loss.

If a pixel has a visibility value of $0$, its contribution to the masked image is zero.

Consider this example:

\\[ I_R= \begin{bmatrix} 10&20&30 \end{bmatrix} \\]

\\[ \hat I_R= \begin{bmatrix} 12&25&80 \end{bmatrix} \\]

\\[ M\_{IR}= \begin{bmatrix} 1&1&0 \end{bmatrix} \\]

Applying the mask gives:

\\[ I_R\odot M\_{IR} = \begin{bmatrix} 10&20&0 \end{bmatrix} \\]

\\[ \hat I_R\odot M\_{IR} = \begin{bmatrix} 12&25&0 \end{bmatrix} \\]

The third pixel is excluded from the image loss because its mask value is zero.

This prevents a transient object at that pixel from forcing the 3DGS model to reconstruct its appearance.

## 3. Mask regularization loss

The paper defines the mask regularization term as:

\\[ \boxed{ \mathcal{L}\_M=(1-M\_{IR})^2 } \tag{12} \\]

This is the equation as supplied. In practice, a mask loss is usually aggregated over pixels, for example:

\\[ \mathcal{L}\_M= \frac{1}{N}\sum\_{i=1}^{N}(1-M\_{IR,i})^2 \\]

where $N$is the number of pixels.

### What does this equation do?

Assume a pixel's predicted visibility is $M\_{IR,i}$.

- If $M\_{IR,i}=1$, its regularization penalty is $0$.
- If $M\_{IR,i}=0.5$, its penalty is $0.25$.
- If $M\_{IR,i}=0$, its penalty is $1$.

Minimizing this penalty encourages the mask to retain pixels rather than hiding them.

This is important because, without regularization, the network could simply predict an all-zero mask. Then, the image loss could become uninformative because the model would hide all reconstruction errors.

## 4. Combined training objective

The two losses serve different purposes:

\\[ \mathcal{L}\_I \quad\longrightarrow\quad \text{Match the rendered image to the reference} \\]

\\[ \mathcal{L}\_M \quad\longrightarrow\quad \text{Discourage excessive masking} \\]

A natural combined objective is:

\\[ \boxed{ \mathcal{L}\_{\text{total}} = \mathcal{L}\_I+\lambda_M\mathcal{L}\_M } \\]

where $\lambda_M$controls the strength of mask regularization. This combined equation illustrates the usual approach; the excerpt you provided does not specify the coefficient or the exact combined objective.

## 5. Why does the paper introduce a mask threshold?

The predicted visibility mask may assign continuous values between $0$and $1$. A threshold $\tau$can turn those values into a binary mask:

\\[ M\_{IR,i}^{\text{binary}}= \begin{cases} 1, & M\_{IR,i}\geq\tau\\\ 0, & M\_{IR,i}<\tau \end{cases} \\]

- Lower threshold: More pixels are retained, including potentially transient objects.
- Higher threshold: More pixels are rejected, but some useful static-object pixels may also be removed.

The threshold controls the trade-off between excluding transient objects and preserving the appearance of static objects.

$$
\begin{array}{|c|l|}
\hline
\text{Mathematical Component} & \text{Purpose} \\
\hline
M_{IR}=U_{\theta}^{2D}(I_R)
& \text{Predict the visibility mask} \\
\hline
\mathcal{L}_I
& \text{Compare reference and rendered images in retained regions} \\
\hline
\mathcal{L}_M
& \text{Discourage the mask from hiding meaningful pixels} \\
\hline
\text{Threshold } \tau
& \text{Control which pixels are retained or excluded} \\
\hline
\end{array}$$


The core idea: Wild-GS learns which image regions are useful for reconstructing a consistent 3D scene. It focuses rendering supervision on static objects and reduces the influence of transient objects, while regularization discourages the mask from simply hiding everything.

The main point of Section 4.4 is how Wild-GS combines its losses and why it does not activate every technique from the beginning.

## 1. Training objective

 $$ \boxed{ \mathcal L\_{\text{total}} = \mathcal L_I+ \lambda_M\mathcal L_M+ \lambda_D\mathcal L_D } $$

| Term                      | Purpose                                                       |
| ------------------------- | ------------------------------------------------------------- |
| \\(\mathcal L_I\\)        | Makes the rendered image match the reference image.           |
| \\(\mathcal L_M\\)        | Trains the model to predict the transient-object mask.        |
| \\(\mathcal L_D\\)        | Encourages rendered depth to agree with estimated depth.      |
| \\(\lambda_M,\lambda_D\\) | Control the relative importance of the mask and depth losses. |

The model optimizes the combined objective to improve image appearance, transient-object handling, and 3D geometry.

## 2. Why are some techniques disabled initially?

The predicted mask \\(M\_{I_R}\\) is inaccurate during early training. Two techniques depend heavily on this mask:

- Depth regularization: uses the mask to exclude transient objects from depth supervision.
- Explicit appearance control: uses the mask to select valid image pixels for back-projecting them into 3D and constructing triplane features.

If the mask incorrectly identifies a person as static scene content, for example, that person's depth and appearance could contaminate the reconstructed scene.

Therefore, Wild-GS initially avoids these mask-dependent techniques. As training improves the mask prediction, these techniques can be used more reliably.

## 3. Conceptual training stages

Initial stage

 $$ \mathcal L\_{\text{initial}} \approx \mathcal L_I+\lambda_M\mathcal L_M $$

The model focuses on image reconstruction and learning the mask, without the mask-dependent depth regularization and explicit appearance control.

Later stage

 $$ \mathcal L\_{\text{later}} = \mathcal L_I+\lambda_M\mathcal L_M+\lambda_D\mathcal L_D $$

The complete objective includes depth regularization. Explicit appearance control is also introduced as the mask becomes more reliable.

These equations illustrate the training strategy; the paragraph does not specify the exact switching iteration or all implementation details.

In one sentence: Wild-GS first learns a reasonable scene and mask, then uses the more reliable mask to improve depth and appearance modeling without allowing early mask errors to corrupt the 3D reconstruction.

# Wild-GS: Experimental Results
The experiments answer four main questions:
1. Does Wild-GS produce better novel-view renderings than existing methods?
2. Which components actually improve the results?
3. Can Wild-GS train and render faster than NeRF-based approaches?
4. Can it transfer the appearance of a reference image to new viewpoints


**Why reduce the mask-loss weight linearly?**

The mask loss is weighted by $\lambda_M$:

$$ \mathcal L_{\text{total}}
=\mathcal L_I+\lambda_M\mathcal L_M+\lambda_D\mathcal L_D$$

The authors gradually reduce $\lambda_M$ during training.
The intuition is that mask supervision is useful for learning to identify transient objects, but the model should eventually prioritize reconstructing the persistent scene rather than continuing to optimize the mask as strongly.
A linear schedule can be written as:

$$ \lambda_M(t)=
\lambda_{\text{start}}+
\frac{t}{T}
(\lambda_{\text{end}}-\lambda_{\text{start}})$$

where $t$ is the current training iteration and $T$ is the schedule duration. This illustrates a linear schedule; the excerpt does not provide the exact endpoints.

# Quantitative comparison

$$
\begin{aligned}
\text{PSNR Improvement} &\approx 3\,\text{dB} \\
\text{Training Speedup} &\approx 200\times \\
\text{Rendering Speedup} &\approx 10{,}000\times \\
\text{Wild-GS Training Time} &= 14\,\text{minutes} \\
\text{Training Iterations} &= 15{,}000
\end{aligned}
$$

Comparison: Wild-GS vs. CR-NeRF, as reported by the authors. Wild-GS also achieves the highest reported PSNR and SSIM among the compared methods.

**3. Qualitative comparison: Why does Wild-GS look better?**


The authors compare rendered images in Figure 4.

The paper highlights two types of improvement:

- Appearance: Better handling of different lighting, exposure, and color tones between photographs.
- Geometry: More accurate structures and fewer missing or distorted regions.


**Why does this happen?**


Ordinary 3DGS excels at efficient rendering but does not explicitly model the appearance variations commonly found in unconstrained tourist photographs.

Wild-GS adds global and local appearance embeddings, intrinsic Gaussian features, depth regularization, and transient-object handling to address these problems.

The images above provide scene context; they are not the actual comparison panels from Figure 4.

**4. Ablation study: What happens when components are removed?**


An ablation study removes one component at a time to test its contribution.


**A. Remove depth regularization**

Observed problem: missing or incorrect geometry
Depth regularization constrains the depth map so that back-projected reference pixels align with the correct 3D regions.

Without it, the local appearance features may be associated with incorrectly positioned Gaussians.

Result: geometry can be missing or distorted, and rendering quality degrades.


Remember the earlier equation:
$$ \mathbf p_c=dK^{-1}\tilde{\mathbf u} $$

If the depth $$ d $$  is inaccurate, the reconstructed 3D point $$ \mathbf p_c $$  is inaccurate too. That error affects where appearance information is assigned in 3D.

**B. Remove the transient mask**


Observed problem: ghosting and color inconsistencies

People and other temporary objects may appear at different positions in different photographs.

Without the mask, their pixels can contaminate scene reconstruction and appearance features.


Result: ghost-like objects and inconsistent colors, particularly in regions heavily occluded by transient objects.


**C. Remove global appearance encoding**

Observed problem: weak global color-tone transfer

The local triplane already captures substantial appearance information for the reference viewpoint.
Therefore, removing the global embedding may not cause a large reduction in ordinary image-quality metrics.

However, the model struggles to transfer a coherent global color tone to another viewpoint or appearance condition.

**Why can global appearance encoding matter even if PSNR barely improves?**


- This is a key observation in the paper.

- The triplane is constructed from reprojected reference-image colors.

- Consequently, it already contains considerable local appearance information.

- The global embedding may add little to the pixel-level score for the reference appearance.

- But global appearance encoding captures scene-wide characteristics, such as overall tone and illumination.

- Those characteristics become important when rendering from a different viewpoint or transferring a different appearance.


- A component can improve generalization and robustness without producing a large improvement in a particular image-quality metric.

**5. Why does the reverse-direction triplane projection help?**


Recall that Wild-GS creates color triplanes by projecting 3D points onto the XY, YZ, and ZX planes.


A single projection loses information along one axis. Multiple 3D points at different depths may project to the same 2D position.


The method therefore adds opposite-direction projections:

$$ \{c_{xy},c_{yz},c_{zx}\}
\oplus
\{c'_{xy},c'_{yz},c'_{zx}\} $$

These six color maps are fed into a U-Net to produce spatial feature maps.


The ablation study finds that these additional projections improve rendering quality, with a small training-efficiency trade-off.


**Intuition:** More projection directions help the representation retain useful appearance information that a single projection can lose.


**What about cropping the triplane?**

The authors also restrict triplane sampling to the region containing most Gaussians, using the AABB strategy discussed earlier.

Reported outcome:
- Training time decreases by around 39%.
- Rendering quality is preserved or even improved.


This is an efficiency optimization: it avoids spending unnecessary computation sampling feature maps in regions that contain few or no relevant Gaussians.

**6. Appearance transfer — Section 5.3**


Appearance transfer asks a different question from ordinary novel-view synthesis.


- Novel-view synthesis: Render the scene from a new camera viewpoint.
- Appearance transfer: Change the scene's overall appearance using a reference image while retaining its 3D structure.
- Style transfer: Transfer broader visual characteristics, such as warm/cool tones or a different photographic style.


Wild-GS supports appearance transfer by extracting appearance embeddings from reference images.


Suppose two reference images produce global appearance embeddings $\mathbf E^g_A$ and $\mathbf E^g_B$. A blended embedding can be formed as:

$$ \boxed{
\mathbf E^g_{\text{blend}}
=
\alpha\mathbf E^g_A+
(1-\alpha)\mathbf E^g_B
}$$

where $0\leq\alpha\leq1$.
- $\alpha=1$: use appearance A.
- $\alpha=0$: use appearance B.
- $\alpha=0.5$: combine both equally.


This equation illustrates the linear blending principle described in the paper.

The excerpt does not specify that every appearance feature or network input is blended using exactly this formula.


The model can then render novel views with the blended appearance.

The authors report that Wild-GS produces more consistent color transfer than CR-NeRF for the tested reference images.


# Final conclusion of the paper

Wild-GS retains the efficient Gaussian-based representation of 3DGS while adding explicit mechanisms to handle appearance variation and transient objects.


Its main contributions work together:



*   Global appearance embedding	Captures scene-wide color and illumination characteristics.
*   Local triplane embedding	Captures spatially varying appearance.
*   Intrinsic Gaussian feature	Learns stable, material-related appearance characteristics.
*   Depth regularization	Improves geometric consistency and back-projection.
*   Transient mask	Reduces ghosting and contamination from temporary objects.
*   Appearance blending	Enables controllable appearance transfer to novel views.
*   AABB-based sampling	Reduces training cost.

